---
authors:
  - edesz
date: 2026-05-05
---

# Evaluation

## About

In this step, a ML pipeline consisting of the best choice of features, feature pre-processing and ML model found from the validation phase will be evaluated on the test split data which was not seen during validation.

As discussed in the project scope, the [primary evaluation metric for model evaluation](../references/scope/06_analysis.md#choice-of-metrics) is the F2-score and the secondary metric is the recall. Briefly, the F2-score is important since it prioritizes recall over precision, which makes it ideal for this use-case since identifying churned customers is more important than avoiding false alarms (predicting a customer would churn when they actually did not churn). It places twice as much weight on recall as precision. This ensures the model focuses on minimizing False Negatives (predicting no churn for customers who did cancel their credit card services with the bank).

[Model evaluation is performed](../references/scope/06_analysis.md#model-evaluation) on the test data split. Interpretable metrics are used to check if the model can beat a model that makes random predictions. Concept drift is used to check if the model can generalize to data it has not seen during *training* without overfitting to the *training* data. This check for model quality is performed using ML scoring metrics and requires knowledge of the true labels (customer outcome), which we have in our historical dataset. Concept drift is also checked without using the true outcome by comparing the distribution of the predicted probabilities relative to the *training* data. Finally, data drift is also checked to see if the customer data in the test split has drifted relative to that in the *training* data, using [statistical tests per column](#column-statistical-tests) depending on the type of column (continuous for numerical features and discrete for categorical and ordinal features).

Since we only have access to historical customer data, we will use the combined training+validation data split as the *training* (or reference) data. More details about this are discussed later when discussing concept drift using [ML metrics to assess model quality](#dataset-prediction-drift) and [predicted probabilities to model prediction drift](#dataset-prediction-drift).

:::{note}
### Outputs

Artifacts of the evaluation Metaflow flow run are stored localy in the `notebooks/.metaflow/EvaluationFlow/` directory. Nothing is exported to the R2 bucket.
:::

## Python Imports

The Python modules required for this notebook are imported below

In [ ]:
import os
from pathlib import Path

import altair as alt
import evidently.metrics as em
import pandas as pd
from dotenv import load_dotenv
from evidently import BinaryClassification, DataDefinition, Dataset
from great_tables import GT, loc, md, style
from metaflow import Flow, Run

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the required custom modules for plotting and running tests for data drift

In [ ]:
import cc_churn.evaluation as ev
import cc_churn.visualization as vzu
import cc_churn.viz_altair as vzau
from cc_churn.ml_monitor import run_column_data_tests
from utils.display_utils import pygments_highlight

## User Inputs

Below we define variables that will be used later to run the Metaflow evaluation flow

In [ ]:
# R2 data bucket details
prefix = "cloud-run"
r2_key_train = f"{prefix}/train_data.parquet.gzip"
r2_key_val = f"{prefix}/validation_data.parquet.gzip"
r2_key_test = f"{prefix}/test_data.parquet.gzip"

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "gender": "string[pyarrow]",
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

# evaluation
primary_metric_eval = "f2"
threshold_overfit = 5

:::{attention}
Except for `primary_metric_eval` and `threshold_overfit`, every parameter from the above cell is also a parameter of the Metaflow validation flow.

The following parameters do not change from one run of the evaluation flow to another

1. `r2_key_train`
2. `r2_key_val`
3. `r2_key_test`
4. `dtypes_ordinals`
5. `dtypes_categoricals`
:::

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

Below we extract the scores per experiment run and ML model name from validation

In [ ]:
df_best_val_run = (
    pd.concat(
        [
            (
                run.data.df_cv.assign(run_id=run.id)
                .groupby(["run_id", "model_name"], as_index=False)
                .agg({"test_prauc": "mean"})
            )
            for k, run in enumerate(list(Flow("ValidationFlow").runs()), 1)
        ]
    )
    .nlargest(2, "test_prauc")
    .query("model_name == 'HistGradientBoostingClassifier'")
)

We'll now get the [best outputs from the validation phase](./05_get_best_validation_experiment.ipynb#best-validation-outputs)

In [ ]:
best_experiment_run_id = df_best_val_run["run_id"].squeeze()
best_model_name = df_best_val_run["model_name"].squeeze()

Below we construct the full path to the best run of the Metaflow validation flow

In [ ]:
best_run_val = Run(f"ValidationFlow/{best_experiment_run_id}")

## Evaluation

The Metaflow flow for evaluation contains the following steps

1. `start`
   - extract best end-to-end pipeline, features, decision threshold and Metaflow run object determined in the validation phase, from the Metaflow validation flow
2. `extract`
   - load test data from R2 bucket and separate features (`X_test`) from class labels (`y_test`)
   - from best Metaflow run, extract features (`X_train`) from class labels (`y_train`) used during model validation
3. `predict_proba`
   - predict probabilities for
     - all training data used in model validation
     - test data
4. `predict`
   - convert predicted probabilities into hard labels
5. `score`
   - score predictions
6. `gather`
   - combine true and predicted labels in test data
7. `fit_all`
   - train best pipeline using all available data in preparation for making inference predictions

(evaluation-flow)=
The Metaflow evaluation flow is defined below

In [ ]:
import json
import os

import boto3
import pandas as pd
import r2.io_utils as r2io
from cc_churn.evaluation import score_predictions
from cc_churn.mflow_utils import get_metaflow_run_artifacts
from cc_churn.scoring import get_scorers
from metaflow import FlowSpec, NBRunner, Parameter, step


class EvaluationFlow(FlowSpec):
    r2_keys = Parameter(
        name="r2_keys", help="R2 keys", default="{'test': 'E'}"
    )
    dtypes_ordinals = Parameter(
        name="dtypes_ordinals",
        help="ordinal datatypes",
        default="{'B': 'string[pyarrow]'}",
    )
    dtypes_categoricals = Parameter(
        name="dtypes_categoricals",
        help="categorical datatypes",
        default="{'G': 'string[pyarrow]'}",
    )
    best_model_name = Parameter(
        name="best_model_name",
        help="best model name from validation",
        default="LogisticRegression",
    )
    best_experiment_run_id = Parameter(
        name="best_experiment_run_id",
        help="best Metaflow Run ID",
        default="138473443",
    )

    @step
    def start(self):
        self.primary_metric_eval = "f2"
        self.metrics_list_eval = ["f2", "recall"]
        self.tpre, _, self.pipe, self.features, self.threshold, self.run = (
            get_metaflow_run_artifacts(self.best_experiment_run_id)
        )
        self.next(self.extract)

    @step
    def extract(self):
        s3_client = boto3.client(
            "s3",
            endpoint_url=(
                f"https://{os.getenv('ACCOUNT_ID')}.r2.cloudflarestorage.com"
            ),
            aws_access_key_id=os.getenv("ACCESS_KEY_ID_USER2"),
            aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY_USER2"),
            region_name="auto",
        )
        df_test = (
            r2io.pandas_read_parquet_r2(
                s3_client=s3_client,
                bucket_name=os.getenv("BUCKET_NAME"),
                r2_key=json.loads(self.r2_keys)["test"],
            )
            .astype(json.loads(self.dtypes_ordinals))
            .astype(json.loads(self.dtypes_categoricals))
        )

        self.X_train = self.run.data.X
        self.y_train = self.run.data.y
        df_train = self.run.data.X.assign(is_churned=self.run.data.y)

        self.X_test = df_test.drop(columns=["is_churned"])
        self.y_test = df_test["is_churned"]

        df = pd.concat([df_train, df_test])
        self.X = df.drop(columns=["is_churned"])
        self.y = df["is_churned"]
        self.next(self.pred_proba)

    @step
    def pred_proba(self):
        self.y_train_pred_proba = pd.Series(
            self.pipe.predict_proba(self.X_train)[:, 1],
            index=self.X_train.index,
            dtype="float64[pyarrow]",
        )
        self.y_test_pred_proba = pd.Series(
            self.pipe.predict_proba(self.X_test)[:, 1],
            index=self.X_test.index,
            dtype="float64[pyarrow]",
        )
        self.next(self.predict)

    @step
    def predict(self):
        self.y_train_pred = self.y_train_pred_proba >= self.threshold
        self.y_test_pred = self.y_test_pred_proba >= self.threshold
        self.next(self.score)

    @step
    def score(self):
        self.df_scores = score_predictions(
            get_scorers(self.metrics_list_eval),
            self.y_train,
            self.y_train_pred,
            self.y_test,
            self.y_test_pred,
            self.best_model_name,
            self.primary_metric_eval,
            5,
        )
        self.next(self.gather)

    @step
    def gather(self):
        self.df_test_pred = pd.concat(
            [
                self.X_test.assign(
                    y_pred=self.y_test_pred,
                    y_pred_proba=self.y_test_pred_proba,
                ),
                self.y_test,
            ],
            axis=1,
        )
        self.next(self.fit_all)

    @step
    def fit_all(self):
        _ = self.pipe.fit(self.X, self.y)
        self.next(self.end)

    @step
    def end(self):
        pass


_ = NBRunner(EvaluationFlow, pylint=False).nbrun(
    r2_keys=json.dumps({"test": r2_key_test}),
    dtypes_ordinals=json.dumps(dtypes_ordinals),
    dtypes_categoricals=json.dumps(dtypes_categoricals),
    best_model_name=best_model_name,
    best_experiment_run_id=best_experiment_run_id,
)

```{attention}
Similar to the [validation flow](./04_run_validation_experiments.ipynb#validation-flow), the `start()` step only performs lightweight setup. For evaluation, this step also loads the best trained pipeline found from validation. Since it is already trained on all available training data (training+validation data split), it is ready for use to make predictions on the evaluation data split.
```

```{important}
The `fit_all()` step trains the ML model on all available data (combined train, validation and test split). This way the trained model can be directly used to make inference predictions without training. This is similar to the [approach used in the validation flow](./04_run_validation_experiments.ipynb#train-all-validation) where the [`.fit()` step trained a model on all available combined training and validation data](./04_run_validation_experiments.ipynb#validation-flow).
```

```{hint} Single Run of Evaluation Flow
The Metaflow evaluation flow only runs once, with a fixed set of parameters, to evaluate the performance of the best combination of ML model, features and feature preprocessing from validation on the evaluation data set.
```

Get all runs of the Metaflow evaluation flow

In [ ]:
df_eval_flow_runs = pd.DataFrame.from_records(
    [
        {
            "id": run.id,
            "started_at": run.created_at,
            "finished_at": run.finished_at,
            "tags": list(run.tags),
            "pathspec": run.pathspec,
            "finished": run.finished,
            "was_successful": run.successful,
        }
        for run in list(Flow("EvaluationFlow").runs())
    ]
)

These are shown below

In [ ]:
gt = (
    GT(df_eval_flow_runs)
    .tab_header(md("**Evaluation Flow Runs**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["id"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["finished", "was_successful"]),
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

Get the most recent Metaflow evaluation flow run

In [ ]:
df_eval_run = df_eval_flow_runs.sort_values(
    by=["finished_at"], ascending=False
).head(1)

This run is shown below

In [ ]:
gt = (
    GT(df_eval_run)
    .tab_header(md("**Most Recent Run of Evaluation Flow**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["id"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["finished", "was_successful"]),
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

Verify it was completed successfully

In [ ]:
assert not df_eval_run.empty
assert df_eval_run["finished"].squeeze() == True
assert df_eval_run["was_successful"].squeeze() == True

(best-evaluation-outputs)=
Extract the following from the most recent successfully completed run of the Metaflow evaluation flow

1. run ID
2. run object
3. best decision threshold that was determined during validation and used in the evaluation flow
4. all used training+validation data, including the true and predicted class labels and predicted probabilities
   - in the `gather()` step at the end of the flow in the validation phase, the best model was trained on this data
5. test data, including the true and predicted class labels and predicted probabilities
   - in the `pred_proba()` step at the start of the flow in the evaluation phase, the best trained model was used to make predictions on this data

In [ ]:
# get ID of required Metaflow evaluation flow run
run_eval_id = df_eval_run.query(
    "(finished == True) & (was_successful == True)"
)["id"].squeeze()

# get required Metaflow evaluation flow run object
run_eval = Run(f"EvaluationFlow/{run_eval_id}")

# extract decision threshold detremined during the validation phase, from the
# evaluation flow run
best_decision_threshold = run_eval.data.threshold

# extract training features, true and predicted class labels and predicted
# probabilities, from the evaluation flow run object
df_train_val_pred = pd.concat(
    [
        run_eval.data.X_train,
        run_eval.data.y_train.rename("y_true"),
        run_eval.data.y_train_pred_proba.rename("y_pred_proba"),
        run_eval.data.y_train_pred.rename("y_pred").astype(int),
    ],
    axis=1,
)

# extract test features, true and predicted class labels and predicted
# probabilities, from the evaluation flow run object
df_test_pred = run_eval.data.df_test_pred

### Evaluate Model Performance on Test Split

#### Interpretable Metrics

We will begin [evaluation of the performance of the best model](../references/scope/06_analysis.md#model-evaluation) on unseen (test split) data using two interpretable metrics - cumulative uplift and gain. There are three main benefits to these

1. they are easy to understand
   - plots of this metric make it easy to demonstrate to a non-technical stakeholder why the model should be deployed
2. they measure how much better the best ML model performs compared to random targeting
   - this directly translates model quality into business profit and allows us to determine if the model provides value
3. they do not use the predicted labels (`y_pred`) so they do not rely how well the best decision threshold (`best_decision_threshold`) was [optimized during the validation phase](./04_run_validation_experiments.ipynb)
   - if the threshold was not fully optimized, then we could still quantitatively assess model performance using these two metrics

The first metric is the uplift.

We will get the uplift by sorting customers in descending order of their probability score and then segmenting them into 20 ventiles. Next, we calculate the uplift as the ratio of the churn rate in the top *N*% of the test data to the churn rate across all customers in the test data. This is done in a cumulative manner by increasing *N* from 5% to 100% in steps of 5% and calculating the uplift. Then we calculate the slope as the absolute percent difference between successive rows.

This is shown below

In [ ]:
df_uplift = ev.calculate_cumulative_uplift(
    df_test_pred, "is_churned", "y_pred_proba", steps=20
)

Next, we auto-detect the visually meaningful elbow is actually where the rate of decline in uplift sharply changes. This is where the slope rapidly increases from nearly flat (zero slope) to steep (a large non-zero value). This will detect the point with the largest change in slope (discrete second derivative).

:::{note} Elbow Extraction Logic
:class: dropdown
:open: true
The objective is to identify the *elbow point* in the cumulative uplift curve. This is the percentile at which the uplift begins to decline much more rapidly, indicating diminishing returns from targeting additional customers.

The first derivative measures the rate of change in uplift (slope of the curve) between consecutive percentile intervals.

We can approximate slope as

```{math}
\text{slope}_i
=
-\frac{y_{i+1} - y_i}{x_{i+1} - x_i}
```

where $x_i$ is the percentile and $y_i$ is the cumulative uplift at percentile $x_i$.

The negative sign is applied because uplift decreases as percentile increases. This converts the slopes into positive values so that larger values correspond to steeper declines.

For example

1. between 0.05 and 0.10, the slope is small, indicating the uplift is relatively stable
2. between 0.10 and 0.15, the slope increases sharply, indicating the curve is beginning to decline rapidly

This transition is the indicator of the elbow.

The second derivative measures how quickly the slope itself changes between successive intervals.

This can be approximated as

```{math}
\Delta \text{slope}_i
=
\text{slope}_{i+1} - \text{slope}_i
```

Large positive values indicate that the curve suddenly becomes much steeper.

Finally, the elbow is identified as the point corresponding to the maximum increase in slope

```{math}
\text{elbow index}
=
\arg\max_i
\left(
\Delta \text{slope}_i
\right)
```

Because each slope is defined between two consecutive percentile points, the detected elbow corresponds to the percentile immediately after the largest increase in slope.

In cumulative uplift curve, the operationally meaningful elbow is the point where the marginal benefit of targeting additional customers begins to decrease rapidly. So, this second-derivative approach captures this behavior directly by identifying the sharpest acceleration in the decline of uplift.
:::

This is implemented in the following function

In [ ]:
pygments_highlight(
    fpath=str(PROJ_ROOT / "src" / "cc_churn" / "evaluation.py"),
    unwanted_lines=(
        list(range(0, 241)) + list(range(243, 280)) + list(range(305, 369))
    ),
    style="vs",
)

This function is now used to get the co-ordinates of the uplift elbow point

In [ ]:
elbow_percentile, elbow_uplift = ev.get_elbow_point(df_uplift, "uplift")

Next, we will plot the cumulative uplift as a function of *N*

In [ ]:
chart = vzau.plot_uplift_curve(
    df=df_uplift,
    xvar="percentile:Q",
    yvar="uplift:Q",
    xtitle="Population Percentile",
    ytitle="Cumulative Uplift",
    ptitle=alt.TitleParams(
        text=(
            f"Curve Plateau is Detected after Top "
            f"{elbow_percentile * 100:.0f}%, Resulting in an Uplift "
            f"of {elbow_uplift:.2f}"
        ),
        anchor="start",
        dx=50,
        fontSize=18,
    ),
    tooltip=[
        alt.Tooltip("percentile", title="Percentile", format=".2f"),
        alt.Tooltip("uplift", title="Uplift", format=".2f"),
    ],
    plateau=elbow_percentile,
    save_params=dict(
        fpath=figures_dir / "fig_28_eval_cumulative_uplift_curve.html"
    ),
    fig_size=dict(width=700, height=350),
)
chart

**Notes**

1. The vertical line marks the point where targeting additional customers yields diminishing returns (in this case, where the uplift changes by less than 0.05).

**Observations**

1. The plot starts with a sharp curve as many true churners are captured in the first few ventiles. As we increase *N* we are adding more customers with lower predicted probability scores whose outcome could have been *randomly guessed*, the cumulative uplift drops and eventually flattens toward 1.0.
2. From the drop-off point (elbow), we can see that the model is six times more effective than a random model. This is reassuring.
3. The elbow is the exact point where the concentration of churners starts to drop off significantly. Here, this occurs after the top 10% of customers who actually did churn. We would have to contact these customers for maximum return.
4. The 10% uplift is not higher than the 5% uplift. This is also reassuring since it suggests the model performs well at ranking the most certain cases of churn correctly at the very top.

Next, this is done for cumulative gain in a similar manner.

We first sort customers in descending order of their predicted probability scores and segment them into 20 ventiles. We then calculate the gain as the ratio of the actual churners found in the top *N*%) of the test data to the total number of churners in the entire dataset. Similar to uplift, this is again performed cumulatively by increasing *N* from 5% to 100% in steps of 5%, representing the percentage of all potential churners we successfully capture as we expand our target list.

The cumulative gain is extracted below

In [ ]:
df_gain = ev.calculate_cumulative_gain(
    df_test_pred, "is_churned", "y_pred_proba", steps=20
)

The co-ordinates of the uplift elbow point are extracted

In [ ]:
elbow_percentile, elbow_gain = ev.get_elbow_point(df_gain, "gain")

Finally, the cumulative gain is shown in a plot against *N*

In [ ]:
chart = vzau.plot_gain_curve(
    ev.calculate_cumulative_gain(
        df_test_pred, "is_churned", "y_pred_proba", steps=20
    ),
    xvar="percentile:Q",
    yvar="gain:Q",
    xtitle="Population Percentile",
    ytitle="Cumulative Gain (Fraction of Positives Found)",
    ptitle=alt.TitleParams(
        text=(
            "Contact Top 20% of At-Risk Customers to Capture the Top 90% of "
            "True Churners"
        ),
        anchor="start",
        dx=50,
        fontSize=18,
    ),
    tooltip=[
        alt.Tooltip("percentile", title="Percentile", format=".2f"),
        alt.Tooltip("gain", title="Gain", format=".2f"),
    ],
    plateau=elbow_percentile,
    save_params=dict(
        fpath=figures_dir / "fig_29_eval_cumulative_gain_curve.html"
    ),
    fig_size=dict(width=700, height=400),
)
chart

**Observations**

1. We could capture >90% of all true churners by contacting the top 20% (by predicted risk) of all the customers at risk of churning, in the test data.
2. The model's predictions outperform random guessing, which is shown in the grey dashed line. This is reassuring.

#### Summary

In terms of model performance, both interpretable metrics indicate the best ML model is out-performing random guessing. Uplift indicates outperformance is by a factor of approximately 6.1 and that we would have to contact to top 10% of at-risk customers by risk in order to maximize return.

### ML Metrics

(concept-drift-overfit)=
Next, we will use ML scoring metrics to monitor model quality. Since we have tracked these metrics during the validation phase, we can compare their values between the validation and evaluation phases. If the train and test scores during evaluation are not *within* 10% of the scores during validation then the ML model does not generalize to unseen data and suffers from concept drift. We can also check for model overfitting during the evaluation phase, which is in indication that the model suffers from high variance. During evaluation, if the train score is 10% *higher* than the test score then the model is overfitting and has a high variance.

First, we will get the summary `DataFrame` for the most recent evaluation run object

In [ ]:
df_scores_eval = run_eval.data.df_scores.assign(split="test").rename(
    columns={
        f"pct_diff_{primary_metric_eval}": "pct_diff",
        f"is_overfit_{primary_metric_eval}": "is_overfit",
        f"is_overfit_significant_{primary_metric_eval}": (
            "is_overfit_significant"
        ),
    }
)

This is shown below

In [ ]:
gt = (
    GT(df_scores_eval)
    .tab_header(
        md("**Metrics and Metadata from Most Recent Evaluation Flow Run**")
    )
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_eval}"]),
    )
    .tab_style(
        style=style.fill(color="cyan"),
        locations=loc.body(columns=[f"train_{primary_metric_eval}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .fmt_number(
        columns=[
            "train_f2",
            "train_recall",
            "test_f2",
            "test_recall",
            "pct_diff",
        ],
        decimals=3,
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

Now, we get the same metrics and metadata during validation

In [ ]:
df_scores_val = (
    best_run_val.data.df_cv.assign(split="val")
    .drop(columns=["fit_time", "score_time", "estimator", "feat_group"])
    .groupby(["model_name", "split"], as_index=False)
    .agg(
        {
            f"{s}_{m}": "mean"
            for m in run_eval.data.metrics_list_eval
            for s in ["train", "test"]
        }
    )
    .assign(
        pct_diff=lambda df: (
            df[f"train_{primary_metric_eval}"]
            .sub(df[f"test_{primary_metric_eval}"])
            .abs()
            .div(df[f"train_{primary_metric_eval}"])
            .mul(100)
        ),
        is_overfit=lambda df: (
            (
                df[f"train_{primary_metric_eval}"]
                > df[f"test_{primary_metric_eval}"]
            ).astype("bool[pyarrow]")
        ),
        is_overfit_significant=lambda df: (
            (
                (df["is_overfit"] == True)
                & (df["pct_diff"] > threshold_overfit)
            ).astype("bool[pyarrow]")
        ),
    )[list(df_scores_eval)]
)

This is shown below

In [ ]:
gt = (
    GT(df_scores_val)
    .tab_header(md("**Metrics and Metadata from Best Validation Flow Run**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_eval}"]),
    )
    .tab_style(
        style=style.fill(color="cyan"),
        locations=loc.body(columns=[f"train_{primary_metric_eval}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .fmt_number(
        columns=[
            "train_f2",
            "train_recall",
            "test_f2",
            "test_recall",
            "pct_diff",
        ],
        decimals=3,
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

We will now combine the metrics during the validation and evaluation phases

In [ ]:
df_scores_val_eval = pd.concat(
    [df_scores_val, df_scores_eval], ignore_index=True
)

We show this below

In [ ]:
gt = (
    GT(df_scores_val_eval)
    .tab_header(md("**Metrics and Metadata from Validation and Evaluation**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_eval}"]),
    )
    .tab_style(
        style=style.fill(color="cyan"),
        locations=loc.body(columns=[f"train_{primary_metric_eval}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .tab_style(
        style=[
            style.fill(color="grey"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["split"]),
    )
    .fmt_number(
        columns=[
            "train_f2",
            "train_recall",
            "test_f2",
            "test_recall",
            "pct_diff",
        ],
        decimals=4,
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

**Observations**

1. During evaluation (`split = 'test'`), `pct_diff` is positive, which indicates the train score (`f2`, which is the primary metric during evaluation) is higher. This indicates the model is overfitting during both validation and evaluation, but it is within the allowed 10% threshold in both phases. So, the overfitting of the best ML model is not significant.

Next, we will [use the ML metrics to check for concept drift](https://www.evidentlyai.com/ml-in-production/concept-drift#model-quality-metrics) or generalization. As [mentioned above](#concept-drift-overfit), we will compare the average test score from cross-validation during the validation phase (first row) to the test score during evaluation (second row). In order to make this comparison, we will use the primary metric during evaluation (`f2`) used in the evaluation phase. If the evaluation score is not within 10% of the validation score then the model has suffered from concept drift.

(datasets-concept-drift)=
```{note} Follow the Evidently Approach to Evaluate ML Metrics with ML Model Scores
We are following the approach used by the [`evidently` Python library](https://docs.evidentlyai.com/introduction) to check for concept drift using ML model quality metrics. To do this, `evidently`'s definition of a *current* dataset is our test data split and the *reference* dataset is the average of the validation folds during outer CV of nested cross-validation that was used during the validation phase. Here we are only comparing scoring metrics, so the average metric from the outer CV will be used as the *reference* data. The test data split is the held out data that has not been used in model training, so it is treated as live data for this purpose. So, we will treat it as the *current* dataset.
```

These checks are performed below for both the the primary (`f2`) and secondary (`recall`) metric during evaluation

In [ ]:
gt = (
    GT(
        ev.transform_model_metrics(
            df_scores_val_eval.filter(regex="^test_|split|model_"),
            ["f2", "recall"],
        )
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(md("**Concept Drift Tests**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["metric"]),
    )
    .tab_style(
        style=style.fill(color="cyan"),
        locations=loc.body(columns=["description"]),
    )
    .data_color(
        columns="status",
        palette=["green", "darkred"],
        domain=["SUCCESS", "FAIL"],
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["value"]),
    )
    .fmt_number(columns=["value"], decimals=4)
)
gt

**Observations**

1. Both metrics are within their allowed 10% bounds. So, at a 10% threshold, the model does not suffer from concept drift.

### Statistical Tests for Data and Prediction Drift

We will now explicitly use `evidently` to [evaluate the performance of the best ML model](https://www.evidentlyai.com/ml-in-production/model-monitoring) using tests for drift in the [evaluation (unseen) data](https://www.evidentlyai.com/ml-in-production/data-drift) and [model predictions of the evaluation data](https://www.evidentlyai.com/ml-in-production/concept-drift).

```{important} Combine Drift Tests
Since features and prediction columns are in the same `DataFrame`s for the combined train and validation data (`df_train_val_pred`) and test data (`df_test_pred`), tests for both types of drift will be run simultaneously.
```

First, we will extract the ordinal and categorical features from the best evaluation run using the keys in datatypes dictionaries for both types of features

In [ ]:
ordinal_features = list(eval(run_eval.data.dtypes_ordinals))
categorical_features = list(eval(run_eval.data.dtypes_categoricals))

Next, we will programmatically extract the numerical features which will be all other columns excluding the customer identifier column

In [ ]:
numeric_features = list(
    set(list(run_eval.data.X_train))
    - set(["clientnum"] + ordinal_features + categorical_features)
)

Now, we'll create an [`evidently` data definition](https://docs.evidentlyai.com/docs/library/data_definition#tabular-data) in which we specify two types of column to be tested

1. numerical columns
   - this is a combination of the extracted numerical features and the predicted probability of churn (`y_pred_proba`)
2. categorical column 
   - this is a combination of categorical and ordinal features that were extracted above and the true and predicted labels
     - the true label is available since this is historical data but this column would have to be excluded in live data since it would not be available

In [ ]:
data_definition = DataDefinition(
    id_column="clientnum",
    numerical_columns=numeric_features + ["y_pred_proba"],
    categorical_columns=(
        ordinal_features + categorical_features + ["y_pred", "y_true"]
    ),
    classification=[
        BinaryClassification(
            target="y_true",
            prediction_labels="y_pred",
            prediction_probas="y_pred_proba",
            pos_label="1",
        )
    ],
)

Next, we'll [create `evidently` datasets from `pandas.DataFrame`s](https://docs.evidentlyai.com/docs/library/data_definition#basic-flow) for the extracted combined train+validation data and test datahttps://docs.evidentlyai.com/docs/library/data_definition#basic-flow

In [ ]:
# combined train+validation
dataset_train_val = Dataset.from_pandas(
    df_train_val_pred.astype({c: float for c in numeric_features}).astype(
        {"y_pred_proba": float, "y_true": str, "y_pred": str}
    ),
    data_definition=data_definition,
)

# test
dataset_test = Dataset.from_pandas(
    run_eval.data.df_test_pred.rename(columns={"is_churned": "y_true"})
    .astype({c: float for c in numeric_features})
    .astype({"y_pred_proba": float, "y_true": str, "y_pred": str}),
    data_definition=data_definition,
)

(datasets-prediction-drift)=
```{attention} Current and Reference Data for Drift Evaluation
In order [to perform drift detection with `evidently`, two datasets are required](https://learn.evidentlyai.com/ml-observability-course/module-4-designing-effective-ml-monitoring/how-to-choose-reference-dataset-ml-monitoring#why-use-a-reference-dataset): [current data and reference data](https://docs.evidentlyai.com/docs/library/data_definition#special-cases). [This post](https://oleg-dubetcky.medium.com/detecting-data-drift-with-evidently-ai-11bfe378cb00) discusses this further.

We will treat the dataset with the combined training and validation data (`dataset_train_val`) as the *reference data*. This is the data for which our ML model has been trained and for which the outcomes are known.

The test dataset (`dataset_test`) will be the *current data*. For historical data, as is the case in this project, we again know the outcome in this dataset. For live customer data, this is not known.

### Limitation

For our case of using historical data, the combined train and validation data is used as the reference data. But, for live data, [the training data should not be treated as a reference](https://learn.evidentlyai.com/ml-observability-course/module-4-designing-effective-ml-monitoring/how-to-choose-reference-dataset-ml-monitoring#what-makes-a-good-reference-dataset), so the following should be considered

1. for the first batch of live customer data for which churn is to be predicted, [the test data is the reference data](https://learn.evidentlyai.com/ml-observability-course/module-4-designing-effective-ml-monitoring/how-to-choose-reference-dataset-ml-monitoring#what-makes-a-good-reference-dataset)
2. for subsequent batches of live data, [the data from the previous batch should be treated as reference data](https://learn.evidentlyai.com/ml-observability-course/module-4-designing-effective-ml-monitoring/how-to-choose-reference-dataset-ml-monitoring#reference-dataset-for-drift-detection)
```

(column-statistical-tests)=
Next, we will define [`evidently` drift metrics at the column level](https://docs.evidentlyai.com/metrics/all_metrics#data-drift). These metrics are statistical tests on the specified columns to check for data drift between the combined train+validation data and the test data

In [ ]:
metrics = [
    em.ValueDrift(column=c, method="wasserstein", threshold=0.05)
    for c in numeric_features + ["y_pred_proba"]
] + [
    em.ValueDrift(column=c, method="jensenshannon", threshold=0.05)
    for c in ordinal_features + categorical_features + ["y_pred", "y_true"]
]

```{hint} Types of Statistical Tests are Based on Size of Data Splits
We can run tests that use Wasserstein and Jenson-Shannon distances since we have data in **both** splits that have more than 1,000 rows, [as per the `evidently` documentation](https://docs.evidentlyai.com/metrics/customize_data_drift#tabular-data).
```

Next, we'll use a custom Python module to define and then run an [`evidently` `Report`](https://docs.evidentlyai.com/docs/library/report) in order to perform the evaluations of data and prediction drift

In [ ]:
df_drift_tests = run_column_data_tests(
    metrics, dataset_test, dataset_train_val
).assign(
    status_manual=lambda df: (
        (df["value"] > df["threshold"]).map({True: "FAIL", False: "SUCCESS"})
    )
)

```{important} Sanity Check
We have also performed a naive sanity check to manually determine the status of the test. This is stored in the `status_manual` column, which is appended to the output from above, and should be identical to the `status` column.
```

The fraction of passing and failing drift tests is extracted below

In [ ]:
df_drift_summary = (
    df_drift_tests["status"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
    .merge(
        (
            df_drift_tests["status_manual"]
            .value_counts(normalize=True)
            .mul(100)
            .reset_index()
        ),
        left_on=["status"],
        right_on=["status_manual"],
        suffixes=("", "_manual"),
    )
    .drop(columns=["status_manual"])
)

This is shown below

In [ ]:
gt = (
    GT(df_drift_summary)
    .tab_header(md("**Drift Test Outcomes**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["proportion"]),
    )
    .tab_style(
        style=[
            style.fill(color="grey"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["proportion_manual"]),
    )
    .fmt_number(
        columns=["proportion", "proportion_manual"],
        decimals=3,
    )
)
gt

**Observations**

1. It is reassuring that more than 90% of the drift tests have passed. We will assume this level of success to be acceptable and so no mitigating actions are necessary, however we will investigate the failing tests below.
2. The simple manually implemented approach to get the proportion of passing and failing tests agrees with `evidently`'s built-in approach.

The failing tests are shown below

In [ ]:
gt = (
    GT(df_drift_tests.drop(columns=["id"]).query("status == 'FAIL'"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(md("**Failing Drift Tests**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["column"]),
    )
    .tab_style(
        style=style.fill(color="cyan"),
        locations=loc.body(columns=["description"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["value", "threshold"]),
    )
    .tab_style(
        style=[
            style.fill(color="darkred"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["status", "status_manual"]),
    )
    .fmt_number(columns=["value", "threshold"], decimals=3)
)
gt

The only failing test on the features is for the `contacts_count_12_mon` (number of customer contacts in last 12 months) column. So, drift was observed in this column in the test data relative to the combined training and validation data.

The predicted class imbalance ([hard labels](https://community.deeplearning.ai/t/week-3-whats-difference-between-hard-predictions-and-hard-labels/465146/2)) has also drifted. Notably, the predicted soft labels (`y_pred_proba`, the predicted probabilities) have not drifted. The hard predicted labels are created using the soft predicted labels and the best classifier decision threshold that we determined during validation (`best_decision_threshold` from above). This suggests the decision threshold could use further optimization.

From the true and predicted hard labels, we will now get the true and predicted class imbalance for the test data split

In [ ]:
df_true_pred_class_imbalance = (
    df_test_pred["y_pred"]
    .value_counts(normalize=True)
    .rename("predicted")
    .to_frame()
).merge(
    (
        df_test_pred["is_churned"]
        .value_counts(normalize=True)
        .rename("true")
        .to_frame()
    ),
    left_index=True,
    right_index=True,
)
df_true_pred_class_imbalance.index = df_true_pred_class_imbalance.index.map(
    {False: "No Churn", True: "Churn"}
)
churn_true = df_true_pred_class_imbalance.loc["Churn"]["true"]
churn_pred = df_true_pred_class_imbalance.loc["Churn"]["predicted"]

This is shown below

In [ ]:
gt = (
    GT(df_true_pred_class_imbalance.reset_index())
    .tab_header(md("**Class Imbalance**"))
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["predicted"]),
    )
    .tab_style(
        style=style.fill(color="papayawhip"),
        locations=loc.body(columns=["true"]),
    )
    .fmt_number(
        columns=["predicted", "true"],
        decimals=3,
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
)
gt

**Observations**

1. The class imbalance in the test split is approximately the same as that in the training split (~84%:16%).
2. It is reassuring that the true and predicted class imbalance (hard labels) are close to each other. Unfortunately, they are not close enough to pass the drift test.
3. The test for prediction drift did not fail so, in addition to true and predicted class labels shown here, the distribution of predicted probabilities did not show drift between the training+validation data and the test data.

Show the class imbalance and distribution of prediction probabilities for the test data

In [ ]:
%%time
vzu.plot_class_imbalance_proba_distribution(
    df_class_imbalance=df_true_pred_class_imbalance.rename(columns=str.title),
    df_probabilities=(df_test_pred['y_pred_proba']*100),
    ptitle1='Similar True & Predicted Churn in Test Split',
    title1_xloc=-0.3,
    ptitle2=(
        'Predicted Probabilities show Right Skew with Weak Peak Above ~90%'
    ),
    vline_label=f'Optimized Churn Cutoff ({best_decision_threshold*100:.0f}%)',
    decision_threshold=best_decision_threshold,
    subfigure_width_ratios=[1.15, 3],
    fig_size=(12, 4)
)

```{tip} Best Decision Threshold for Churn versus No Churn
The dashed line shows the optimized decision threshold. Predicted probabilities below this threshold are labeled as the majority class (no churn) and those above the threshold are the minitory class (churn).
```

**Observations**

1. As expected from the predicted class imbalance, the distribution of predicted probabilities is right-skewed and a small fraction of customers have a predicted probability above 50% (the tuned classification decision threshold).

## Conclusions

ML model scores for training and unseen data during the evaluation phase with the test data split are similar to those in the five cross-validation validation folds during the validation phase. This suggests the model can generalize to customers data it has not seen before and does not suffer from concept drift.

Overfitting is present during validation and evaluation but is insignificant, which suggests variance is low. This suggests the model has successfully learned the underlying patterns rather than simply memorizing the majority class (non-churned customers) or noise in the training data.

The model is six times more effective than a random model. So, it is not randomnly guessing the at-risk customers.

Concept drift is also not observed using statistical tests on the predicted probabilities (soft labels) for the test data. 90% of drift checks on the test data passed. Data drift is observed in one of the ML model's features. Also, the predicted hard labels showed weak drift.

Overall, these findings suggest the best model found using cost-sensitive learning is a reliable model that can maintain balanced performance by accurately identifying both the churned and non-churned credit card customers on new, unseen customer data. In conclusion, we can rely on the predictions made by the best ML mode to make predictions from which we extract targeting recommendations.